In [ ]:
import os
import random
import warnings
import numpy as np
import csv
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import r2_score
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as GeoDataLoader
from torch_geometric.nn import GCNConv, global_mean_pool


def trusted_torch_load(path, map_location=None):
    """Load trusted local graph data or model checkpoints across PyTorch versions."""
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


class GCNTeacher(nn.Module):
    def __init__(self, node_dim, global_dim, hidden_dims=None, dropout=0.2):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [128, 128]

        self.norm = nn.BatchNorm1d(node_dim)
        self.convs = nn.ModuleList()
        in_dim = node_dim
        for h in hidden_dims:
            self.convs.append(GCNConv(in_dim, h))
            in_dim = h

        self.dropout = nn.Dropout(dropout)
        if global_dim:
            self.global_norm = nn.BatchNorm1d(global_dim)
            self.global_mlp = nn.Sequential(
                nn.Linear(global_dim, 128),
                nn.ReLU(),
                nn.Dropout(dropout),
            )
            self.final_dim = hidden_dims[-1] + 128
        else:
            self.final_dim = hidden_dims[-1]

        self.output = nn.Sequential(
            nn.Linear(self.final_dim, self.final_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(self.final_dim // 2, 1),
        )

    def forward(self, data):
        x = self.norm(data.x)
        for conv in self.convs:
            x = F.relu(conv(x, data.edge_index))
            x = self.dropout(x)

        x = global_mean_pool(x, data.batch)
        if hasattr(data, "u") and data.u is not None:
            u = self.global_norm(data.u)
            u = self.global_mlp(u)
            x = torch.cat([x, u], dim=1)


        return self.output(x).view(-1)


class StackingMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x):
        return self.net(x).view(-1)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def create_geo_loader(graph_list, batch_size=32, shuffle=True):
    data_list = []
    for graph in graph_list:
        data_list.append(
            Data(
                x=graph["x"],
                edge_index=graph["edge_index"],
                u=graph.get("u", None),
                y=graph["y"],
            )
        )
    return GeoDataLoader(data_list, batch_size=batch_size, shuffle=shuffle)


def collect_teacher_predictions(loader, teachers, device):
    """Return teacher predictions and targets on the standardized target scale."""
    all_targets = []
    all_predictions = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            batch_predictions = []

            for teacher in teachers:
                prediction = teacher(batch).view(-1)
                batch_predictions.append(prediction.cpu().numpy())

            all_predictions.append(np.stack(batch_predictions, axis=1))
            all_targets.append(batch.y.view(-1).cpu().numpy())

    return (
        np.concatenate(all_predictions, axis=0),
        np.concatenate(all_targets, axis=0),
    )


def train_stacking(
    train_dir,
    val_dir,
    teacher_paths,
    save_path,
    hidden_dim=64,
    dropout=0.2,
    epochs=300,
    batch_size=64,
    lr=1e-3,
    lr_patience=20,
    es_patience=50,
    seed=42,
):
    """Train a five-teacher stacking model using validation-based checkpoint selection."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    set_seed(seed)


    train_graphs = trusted_torch_load(os.path.join(train_dir, "graph_data.pt"))
    val_graphs = trusted_torch_load(os.path.join(val_dir, "graph_data.pt"))


    teachers = []
    y_mean_from_ckpt = None
    y_std_from_ckpt = None

    for teacher_path in teacher_paths:
        checkpoint = trusted_torch_load(teacher_path, map_location=device)

        node_dim = checkpoint["node_dim"]
        global_dim = checkpoint.get("global_dim", 0)
        hidden_dims = checkpoint.get("hidden_dims", [128, 128])
        teacher_dropout = checkpoint.get("dropout", 0.2)

        teacher = GCNTeacher(
            node_dim=node_dim,
            global_dim=global_dim,
            hidden_dims=hidden_dims,
            dropout=teacher_dropout,
        ).to(device)
        teacher.load_state_dict(checkpoint["model_state_dict"], strict=False)
        teacher.eval()
        teachers.append(teacher)

        if y_mean_from_ckpt is None and "y_mean" in checkpoint and "y_std" in checkpoint:
            candidate_mean = float(checkpoint["y_mean"])
            candidate_std = float(checkpoint["y_std"])


            if (
                abs(candidate_mean) > 1e-6
                and abs(candidate_std - 1.0) > 0.1
                and candidate_std > 0
            ):
                y_mean_from_ckpt = candidate_mean
                y_std_from_ckpt = candidate_std

    print(f"Loaded {len(teachers)} teacher models.")


    if y_mean_from_ckpt is not None:
        y_mean = y_mean_from_ckpt
        y_std = y_std_from_ckpt
        print(
            "Using target-scaling statistics from teacher checkpoint: "
            f"mean={y_mean:.4f}, std={y_std:.4f}"
        )
    else:
        print("No valid scaling statistics found in teacher checkpoints; using training data.")
        train_targets = torch.stack([graph["y"] for graph in train_graphs]).view(-1)
        y_mean = float(train_targets.mean().item())
        y_std = float(train_targets.std().item()) + 1e-8
        print(f"Training-set statistics: mean={y_mean:.4f}, std={y_std:.4f}")


    for graph in train_graphs:
        graph["y"] = (graph["y"] - y_mean) / y_std
    for graph in val_graphs:
        graph["y"] = (graph["y"] - y_mean) / y_std


    train_geo_loader = create_geo_loader(
        train_graphs, batch_size=batch_size, shuffle=False
    )
    val_geo_loader = create_geo_loader(
        val_graphs, batch_size=batch_size, shuffle=False
    )

    print("Extracting teacher predictions on training set...")
    x_train, y_train = collect_teacher_predictions(
        train_geo_loader, teachers, device
    )
    print("Extracting teacher predictions on validation set...")
    x_val, y_val = collect_teacher_predictions(
        val_geo_loader, teachers, device
    )


    train_dataset = TensorDataset(
        torch.tensor(x_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.float32),
    )
    val_dataset = TensorDataset(
        torch.tensor(x_val, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.float32),
    )

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True
    )

    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False
    )


    input_dim = x_train.shape[1]
    model = StackingMLP(input_dim, hidden_dim, dropout).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=lr_patience,
    )
    criterion = nn.MSELoss()

    best_val_r2 = -np.inf
    best_epoch = 0
    patience_counter = 0
    history = {
        "train_loss": [],
        "val_loss": [],
        "val_r2": [],
    }

    for epoch in range(1, epochs + 1):
        model.train()
        total_train_loss = 0.0

        for features, labels in train_loader:
            features = features.to(device)
            labels = labels.to(device)

            predictions = model(features)
            loss = criterion(predictions, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item() * features.size(0)

        average_train_loss = total_train_loss / len(train_dataset)

        model.eval()
        val_predictions = []
        val_targets = []
        total_val_loss = 0.0

        with torch.no_grad():
            for features, labels in val_loader:
                features = features.to(device)
                labels = labels.to(device)

                predictions = model(features)
                total_val_loss += criterion(predictions, labels).item() * features.size(0)
                val_predictions.append(predictions.cpu().numpy())
                val_targets.append(labels.cpu().numpy())

        average_val_loss = total_val_loss / len(val_dataset)
        val_predictions = np.concatenate(val_predictions)
        val_targets = np.concatenate(val_targets)
        current_val_r2 = r2_score(val_targets, val_predictions)

        history["train_loss"].append(average_train_loss)
        history["val_loss"].append(average_val_loss)
        history["val_r2"].append(current_val_r2)

        scheduler.step(average_val_loss)

        if current_val_r2 > best_val_r2:
            best_val_r2 = current_val_r2
            best_epoch = epoch
            patience_counter = 0

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "y_mean": y_mean,
                    "y_std": y_std,
                    "history": history,
                    "input_dim": input_dim,
                    "hidden_dim": hidden_dim,
                    "dropout": dropout,
                    "best_epoch": best_epoch,
                    "best_val_r2_standardized": best_val_r2,
                },
                save_path,
            )
            print(
                f"Epoch {epoch}: validation R² improved to "
                f"{current_val_r2:.4f}, model saved."
            )
        else:
            patience_counter += 1

        if epoch % 30 == 0:
            print(
                f"Epoch {epoch}: train loss {average_train_loss:.4f}, "
                f"validation loss {average_val_loss:.4f}, "
                f"validation R² {current_val_r2:.4f}"
            )

        if patience_counter >= es_patience:
            print(f"Early stopping at epoch {epoch}")
            break


    return save_path


if __name__ == '__main__':
    seeds = [0, 8, 42, 100, 456, 618, 1189, 2025, 2077, 2048]
    project_root = os.path.abspath(os.getcwd())
    train_dir = os.path.join(project_root, 'data-set', 'train')
    val_dir = os.path.join(project_root, 'data-set', 'validation')
    teacher_order = ('qcut', 'elem', 'molwt', 'fp', 'scaffold')
    teacher_dir = os.path.join(project_root, 'checkpoints', 'teachers')
    teacher_paths = [os.path.join(teacher_dir, f'{name}.pt') for name in teacher_order]
    save_root = os.path.join(project_root, 'results', 'teacher_stacking')
    os.makedirs(save_root, exist_ok=True)


    fixed_config = {
        'hidden_dim': 64, 'dropout': 0.2, 'epochs': 1000,
        'batch_size': 64, 'lr': 1e-3,
        'lr_patience': 20, 'es_patience': 100,
    }
    manifest_path = os.path.join(save_root, 'checkpoint_manifest.csv')
    with open(manifest_path, 'w', newline='', encoding='utf-8') as manifest_file:
        writer = csv.DictWriter(manifest_file, fieldnames=['seed', 'checkpoint_path'])
        writer.writeheader()
        for seed in seeds:
            save_path = os.path.join(save_root, f'stacking_seed{seed}.pt')
            print(f'Training stacking model: seed {seed}.')
            train_stacking(
                train_dir=train_dir, val_dir=val_dir, teacher_paths=teacher_paths,
                save_path=save_path, seed=seed, **fixed_config,
            )
            writer.writerow({'seed': seed, 'checkpoint_path': os.path.relpath(save_path, project_root)})
            manifest_file.flush()
    print(f'Stacking completed. Checkpoint manifest: {manifest_path}')